# Generate answers (Colab)

[Open in Colab](https://colab.research.google.com/github/khacbac/nlp/blob/main/experiment/generate_answers.ipynb)

Answers are saved to **Google Drive** (`MyDrive/nlp-answers/answers_<model>.jsonl`), one line per answer,
so nothing is lost when Colab disconnects or a quota runs out.
**To continue after any error: run the cells again from the top.** Finished answers are skipped;
failed or empty ones are retried.

- Run cells 1-3 every time.
- Gemini and gpt-oss run on a CPU runtime. Qwen needs *Runtime > Change runtime type > T4 GPU*.
- Gemini's free tier allows about 500 requests a day: if it stops with "Quota exhausted", run it again tomorrow.
- Put the keys in Colab *Secrets* (key icon on the left): `GEMINI_API_KEY` (aistudio.google.com),
  `GROQ_API_KEY` (console.groq.com), optionally `OPENROUTER_API_KEY`. Turn on notebook access for each.
- Annotators must not open the answer files: they name the model of each answer.

## 1. Drive + code

In [15]:
from google.colab import drive
drive.mount("/content/drive")

OUT_DIR = "/content/drive/MyDrive/nlp-answers"
REPO = "/content/nlp"

import os
os.makedirs(OUT_DIR, exist_ok=True)
# The Colab copy of the repo is throw-away (answers live on Drive), so always reset it to the latest main.
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} fetch -q origin && git -C {REPO} reset -q --hard origin/main
else:
    !git clone -q https://github.com/khacbac/nlp.git {REPO}
%cd {REPO}/experiment
!git log -1 --format="code at %h: %s"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/nlp/experiment
code at fff1a4a: Send a User-Agent so Groq's Cloudflare does not block requests


## 2. API keys

In [16]:
from google.colab import userdata

for name in ["GEMINI_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY"]:
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: set")
    except Exception:
        print(f"{name}: not found in Secrets (only needed for the model that uses it)")

GEMINI_API_KEY: set
GROQ_API_KEY: set
OPENROUTER_API_KEY: not found in Secrets (only needed for the model that uses it)


## 3. Progress

In [17]:
import json
from pathlib import Path

def progress():
    qids = {json.loads(l)["qid"] for l in open("data/questions.jsonl", encoding="utf-8")}
    for m in ["qwen", "gpt-oss", "gemini"]:
        path = Path(OUT_DIR) / f"answers_{m}.jsonl"
        done = set()
        if path.exists():
            for line in path.open(encoding="utf-8"):
                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if row.get("answer"):
                    done.add(row["qid"])
        print(f"{m:8s} {len(done & qids):4d} / {len(qids)}")

progress()

qwen      500 / 500
gpt-oss   368 / 500
gemini    498 / 500


## 4. Trial: 3 Gemini answers
Read them: Vietnamese, 2-4 sentences, plain prose, faithful to the passage?

In [18]:
!python generate_answers.py --model gemini --limit 3 --out-dir {OUT_DIR}

for line in open(f"{OUT_DIR}/answers_gemini.jsonl", encoding="utf-8").readlines()[:3]:
    row = json.loads(line)
    print(row["qid"], f'({row["num_words"]} words, {row["finish_reason"]})')
    print(row["answer"], "\n")

gemini: 0 to answer, 3 already done -> /content/drive/MyDrive/nlp-answers/answers_gemini.jsonl
0001-0004-0004 (69 words, STOP)
Từ những sai lầm của đoàn Việt Nam như nhờ chuyển điện đài và sử dụng phiên dịch, Trung Quốc đã nắm được toàn bộ các sách lược của phía Việt Nam. Lợi dụng điều này, Trung Quốc tìm cách ngăn chặn các nội dung liên lạc và sử dụng thông tin có được để ép Việt Nam ký hiệp định Geneva theo đúng các lợi ích của họ. 

0001-0007-0001 (82 words, STOP)
Lòng tin của nhân dân vào Đảng giảm sút là do nhiều người có chức, có quyền trong hệ thống tổ chức của Đảng, Nhà nước và các đoàn thể quần chúng đã bị hư hỏng, thoái hóa biến chất. Những người này chỉ chạy theo chức quyền, tiền tài, danh lợi và cá nhân. Tình trạng này đã tác động tiêu cực, đưa đến tình hình nguy kịch đe dọa sự mất còn của chế độ và sự nghiệp cách mạng. 

0002-0015-0001 (86 words, STOP)
Sự thụ phấn tạo ra một cơ sở ổn định nhờ sự hợp nhất của nhân đực thứ hai với nhân cuối cùng. Quá trình này gắn liền với s

## 5. Gemini Flash-Lite (CPU runtime, ~40 min per 500 requests; may take 2 days)

In [19]:
!python generate_answers.py --model gemini --out-dir {OUT_DIR}
progress()

gemini: 2 to answer, 498 already done -> /content/drive/MyDrive/nlp-answers/answers_gemini.jsonl
  [1/2] 0138-0187-0003
  [2/2] 0138-0192-0005
done; 0 still without an answer
qwen      500 / 500
gpt-oss   368 / 500
gemini    500 / 500


## 6. gpt-oss-20b (CPU runtime, ~40 min)
If Groq's free quota runs out, set `PROVIDER = "openrouter"` and rerun.

In [20]:
PROVIDER = "groq"
!python generate_answers.py --model gpt-oss --provider {PROVIDER} --out-dir {OUT_DIR}
progress()

gpt-oss: 132 to answer, 368 already done -> /content/drive/MyDrive/nlp-answers/answers_gpt-oss.jsonl
  [1/132] 0106-0013-0005
  [2/132] 0106-0026-0005
  [3/132] 0106-0029-0005
  [4/132] 0106-0040-0002
  [5/132] 0106-0042-0004
  [6/132] 0106-0061-0005
  [7/132] 0107-0003-0005
  [8/132] 0107-0007-0005
  [9/132] 0107-0012-0001
  [10/132] 0107-0019-0001
  [11/132] 0107-0020-0002
  [12/132] 0107-0023-0004
  [13/132] 0107-0028-0002
  [14/132] 0107-0038-0003
  [15/132] 0107-0044-0005
  [16/132] 0107-0046-0004
  [17/132] 0107-0052-0002
  [18/132] 0107-0055-0001
  [19/132] 0107-0056-0003
  [20/132] 0107-0061-0004
  [21/132] 0107-0064-0001
  [22/132] 0109-0003-0003
  [23/132] 0109-0005-0004
  [24/132] 0109-0019-0003
  [25/132] 0109-0037-0003
  [26/132] 0109-0044-0005
  [27/132] 0110-0010-0002
  [28/132] 0110-0016-0004
  [29/132] 0110-0022-0002
  [30/132] 0112-0008-0004
  [31/132] 0114-0005-0005
  [32/132] 0114-0007-0005
  [33/132] 0114-0010-0003
  [34/132] 0114-0014-0004
  [35/132] 0114-0033-000

## 7. Qwen2.5-7B-Instruct (T4 GPU runtime, ~10 min + ~5 min install)

In [21]:
import subprocess
if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise SystemExit("No GPU: Runtime > Change runtime type > T4 GPU, then run from cell 1 again.")
!pip install -q vllm
!python generate_answers.py --model qwen --out-dir {OUT_DIR}
progress()

qwen: 0 to answer, 500 already done -> /content/drive/MyDrive/nlp-answers/answers_qwen.jsonl
qwen      500 / 500
gpt-oss   500 / 500
gemini    500 / 500


## Next

When all three models show 500 / 500, the person who builds the grading sheets downloads the three
`answers_*.jsonl` files from `MyDrive/nlp-answers` into `experiment/data/` and runs
`python make_grading_sheet.py` (see README).